In [ ]:
import pandas as pd
import numpy as np
import json
import csv
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [80]:
#csv
#json
#excel
#docx

In [81]:
def parse_pdf(file_path: str) -> str:
    
    reader = PdfReader(file_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append(text)
    return "\n\n".join(pages)

In [ ]:
def parse_txt(file_path: str) -> str:
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [ ]:
def parse_markdown(file_path: str) -> str:
    """
    Headers are kept as-is (not stripped) — the chunker's paragraph splitter
    treats blank lines as boundaries, so headers stay glued to the content
    that follows them.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [ ]:
def parse_json(file_path: str) -> str:
    """
    JSON is structured data, not prose — chunking it by raw character count
    would slice through braces and produce garbage for embedding. Instead we
    flatten each record into a sentence-like line. Adjust the flattening
    template below to fit your actual schema.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if isinstance(records, dict):
        records = [records]

    lines = []
    for record in records:
        parts = [f"{key}: {value}" for key, value in record.items()]
        lines.append(", ".join(parts))
    return "\n".join(lines)

In [ ]:
def parse_csv(file_path: str) -> str:
    """
    Same reasoning as JSON — each row gets flattened into one line so the
    chunker's paragraph splitter treats a row as a natural unit rather than
    an arbitrary character slice through the middle of a row.
    """
    lines = []
    with open(file_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            parts = [f"{key}: {value}" for key, value in row.items()]
            lines.append(", ".join(parts))
    return "\n".join(lines)

In [ ]:
def parse_docx(file_path: str) -> str:
    """
    Requires: pip install python-docx
    Paragraphs are joined with double newlines so the chunker's paragraph
    splitter treats each Word paragraph as its own unit, same as it would
    for a PDF page or a markdown paragraph.
    """
    from docx import Document

    doc = Document(file_path)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)

In [82]:
data = parse_pdf("sample.pdf")


In [83]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""],
)

In [88]:
chunks = splitter.split_text(data)

In [ ]:
print(chunks[2])                    # if this shows actual line breaks, data is fine

Home learning Optional acceleration only. The plan must be 
completable inside work hours alone. 
Slippage rule If a cycle slips, extend it — never compress the 
deliverable or skip the checkpoint. A skipped 
checkpoint is a fake checkpoint. 
Logging Every learning block is logged in Orbit as a 
learning task (base 6 pts) so time investment is 
visible next to project work. 
 
Every source below is scoped (which sections/videos, estimated hours). Totals per cycle stay 
inside budget. If you finish early, stretch items are marked ⭐ — they are optional, not silently 
expected. 
Personalization principle for this plan: you are the opposite risk profile from a typical junior AI 
hire — you's built a neural network's forward pass, backprop, and Adam optimizer entirely from
